In [ ]:
#!/usr/bin/env python3
"""
src/stats_d10.py
Implements Statistical Protocol D10:
  - Wilcoxon signed-rank test (alpha = 0.05)
  - Non-parametric Cliff's delta effect size
  - Holm-Bonferroni family-wise error rate correction
  - Automated LaTeX .tex table export for Section V (Results)
"""

import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> tuple[float, str]:
    """Compute non-parametric Cliff's delta and qualitative magnitude."""
    nx, ny = len(x), len(y)
    diffs = np.subtract.outer(x, y)
    delta = (np.sum(diffs > 0) - np.sum(diffs < 0)) / (nx * ny)
    ad = abs(delta)
    if ad < 0.147:
        mag = "negligible"
    elif ad < 0.33:
        mag = "small"
    elif ad < 0.474:
        mag = "medium"
    else:
        mag = "large"
    return float(delta), mag


def holm_bonferroni(p_values: list[float]) -> list[float]:
    """Apply Holm-Bonferroni step-down p-value adjustment."""
    m = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda t: t[1])
    adjusted = [0.0] * m
    prev = 0.0
    for rank, (orig_idx, p) in enumerate(indexed):
        adj = min(1.0, max(prev, p * (m - rank)))
        adjusted[orig_idx] = adj
        prev = adj
    return adjusted


def export_d1_latex_table(df_d1: pd.DataFrame, out_path: str) -> pd.DataFrame:
    """Summarize D1 Predictive Accuracy across 30 runs and export LaTeX table."""
    ref = df_d1[df_d1["model"] == "Proposed_Hybrid_ML"]["f1"].values
    models = ["Static_Threshold", "PIM_Siegmund", "MLP_Neural", "Plain_GBDT", "Proposed_Hybrid_ML"]
    raw_p = []
    rows = []

    for m in models:
        sub = df_d1[df_d1["model"] == m]
        f1_vals = sub["f1"].values
        if m == "Proposed_Hybrid_ML":
            p_val, delta, mag = 1.0, 0.0, "---"
        else:
            _, p_val = wilcoxon(ref, f1_vals)
            delta, mag = cliffs_delta(ref, f1_vals)
        raw_p.append(p_val)
        rows.append({
            "Model": m.replace("_", " "),
            "F1_Mean_SD": f"{sub['f1'].mean():.3f} $\\pm$ {sub['f1'].std():.3f}",
            "Precision": f"{sub['precision'].mean():.3f}",
            "Recall": f"{sub['recall'].mean():.3f}",
            "AUC_ROC": f"{sub['auc_roc'].mean():.3f}",
            "MAPE_Pct": f"{sub['mape_latency'].mean():.2f}\\%",
            "p_raw": p_val,
            "Cliffs_Delta": f"{delta:+.2f} ({mag})" if m != "Proposed_Hybrid_ML" else "---",
        })

    adj_p = holm_bonferroni(raw_p[:-1]) + [1.0]
    for r, p_adj in zip(rows, adj_p):
        r["p_adj"] = "$<0.001$" if p_adj < 0.001 else (f"{p_adj:.3f}" if p_adj < 1.0 else "---")

    summary_df = pd.DataFrame(rows)
    lines = [
        "\\begin{table*}[t]",
        "\\centering",
        "\\caption{Dimension 1 (RQ1): Predictive Performance Across 30 Stochastic Runs (Mean $\\pm$ SD, Holm-Bonferroni Adjusted $p$, and Cliff's $\\delta$).}",
        "\\label{tab:res_d1_predictive}",
        "\\footnotesize",
        "\\begin{tabular}{l c c c c c c c}",
        "\\toprule",
        "\\textbf{Model} & \\textbf{F1-Score} & \\textbf{Precision} & \\textbf{Recall} & \\textbf{AUC-ROC} & \\textbf{Latency MAPE} & \\textbf{Adj. $p$} & \\textbf{Cliff's $\\delta$} \\\\",
        "\\midrule",
    ]
    for _, r in summary_df.iterrows():
        lines.append(
            f"{r['Model']} & {r['F1_Mean_SD']} & {r['Precision']} & {r['Recall']} & "
            f"{r['AUC_ROC']} & {r['MAPE_Pct']} & {r['p_adj']} & {r['Cliffs_Delta']} \\\\"
        )
    lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table*}"])
    with open(out_path, "w") as f:
        f.write("\n".join(lines))
    return summary_df


def export_d3_human_study_table(df_user: pd.DataFrame, out_path: str) -> None:
    """Analyze Practitioner Study (N=12, 72 observations) and export LaTeX table."""
    c1 = df_user[df_user["condition"] == "C1_No_XAI"]
    c2 = df_user[df_user["condition"] == "C2_With_XAI"]

    # Participant-level paired means (N=12) for exact paired Wilcoxon test
    p_c1 = c1.groupby("participant_id")[["diagnostic_accuracy", "decision_time_sec", "likert_trust", "likert_actionability"]].mean()
    p_c2 = c2.groupby("participant_id")[["diagnostic_accuracy", "decision_time_sec", "likert_trust", "likert_actionability"]].mean()

    metrics = [
        ("Diagnostic Accuracy (\\%)", "diagnostic_accuracy", 100.0),
        ("Decision Time (s)", "decision_time_sec", 1.0),
        ("Perceived Trust (1--5)", "likert_trust", 1.0),
        ("Actionability Score (1--5)", "likert_actionability", 1.0),
    ]
    lines = [
        "\\begin{table}[t]",
        "\\centering",
        "\\caption{Dimension 3 Track B (RQ3): Controlled Practitioner Evaluation ($N=12$ Participants, $72$ Task Observations).}",
        "\\label{tab:res_d3_practitioner}",
        "\\footnotesize",
        "\\begin{tabular}{l c c c c}",
        "\\toprule",
        "\\textbf{Metric} & \\textbf{No XAI (C1)} & \\textbf{With XAI (C2)} & \\textbf{$p$-value} & \\textbf{Cliff's $\\delta$} \\\\",
        "\\midrule",
    ]
    for label, col, scale in metrics:
        v1 = p_c1[col].values * scale
        v2 = p_c2[col].values * scale
        _, p_val = wilcoxon(v2, v1)
        delta, mag = cliffs_delta(v2, v1)
        p_str = "$<0.001$" if p_val < 0.001 else f"{p_val:.3f}"
        lines.append(
            f"{label} & {v1.mean():.2f} $\\pm$ {v1.std():.2f} & {v2.mean():.2f} $\\pm$ {v2.std():.2f} & "
            f"{p_str} & {delta:+.2f} ({mag}) \\\\"
        )
    lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table}"])
    with open(out_path, "w") as f:
        f.write("\n".join(lines))